# Practice Day 14 · Window functions and a timed set

Ranks, top-N per group, `lag`, running totals, latest rows, sessions and streaks: Zillacode's *Windows Functions* problems, which are most of its Hard ones. The day ends with a timed set of 6 problems in 65 minutes, like an interview.

14 problems, about 2 hours. They prepare you for the Zillacode problems of Days 15 to 26.

## How to work
* Run the **Setup** cell first. `F` is `pyspark.sql.functions` and `Window` is `pyspark.sql.Window`, the same imports Zillacode's editor gives you (it calls the window class `W`).
* Every problem is in Zillacode's format: complete the function `etl(...)` in the cell under it, so that it **returns** a DataFrame, then run the cell. The last line, `etl(...).show()`, prints your result.
* Your result must look exactly like the **expected output**: same columns, same rows, same order. Where an expected output is left-aligned, it was printed with `.show(truncate=False)`, as the cell already does.
* Stuck for 15 minutes? Open the hint under the problem. The answers are collapsed at the very end of the notebook: open one only after you have tried, then write the solution again yourself without looking.

**Setup (once):** Java 17 or 21 and Python 3.10 or newer. In a new folder: `python3.12 -m venv .venv`, `source .venv/bin/activate`, `pip install pyspark==4.2.0 "pandas>=2.2,<3" "pyarrow>=18" jupyterlab`, then `jupyter lab` and open this notebook.

## The data
The same small online shop as in the theory days. The setup cell creates every table, and you can read every row in it.

* **customers**: *customer_id*, *name*, *email*, *city*, *signup_date*
* **products**: *product_id*, *product_name*, *category*, *price* (dollars for one item)
* **orders**: *order_id*, *customer_id*, *product_id*, *quantity*, *order_date*, *status* (`delivered`, `shipped` or `cancelled`)

Good to know: order 1011 is a guest checkout (no *customer_id*); customer 6 has no email and no city; customer 7 has never ordered; product 107 has never been sold; order 1016 is for product 108, which is no longer in *products*.

This notebook also has:
* **events**: website visits: *customer_id*, *event_time* (a timestamp), *page*; two rows were loaded twice

## Setup
Run this cell once. It starts Spark and creates the tables.

In [ ]:
# Setup: starts Spark and creates the shop tables. Run it once.
import logging, os, shutil, sys, time
from datetime import date
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)   # Spark's Python workers use this notebook's Python
from pyspark.sql import SparkSession, Window, functions as F

spark = (SparkSession.builder
         .master("local[4]")   # 4 tasks at a time on your machine
         .appName("practice14")
         .config("spark.sql.shuffle.partitions", "4")          # small data: 4 shuffle partitions are enough
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.log.level", "FATAL")                  # hide Spark's log messages
         # Spark 4.2.0 bug (SPARK-60294): Python UDFs on Parquet can hang without this
         .config("spark.sql.execution.arrow.pythonUDF.columnarInput.enabled", "false")
         .getOrCreate())
for name in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
    logging.getLogger(name).setLevel(logging.CRITICAL)        # errors still show, without an extra JSON log

customers_rows = [
    # customer_id, name, email, city, signup_date
    (1, "Aziz Karimov",     "aziz@mail.uz",     "Tashkent", date(2024, 1, 5)),
    (2, "Bella Schmidt",    "bella@web.de",     "Berlin",   date(2024, 1, 20)),
    (3, "Carlos Ruiz",      "carlos@correo.es", "Madrid",   date(2024, 2, 2)),
    (4, "Dilnoza Rahimova", "dilnoza@mail.uz",  "Tashkent", date(2024, 2, 14)),
    (5, "Emma Wilson",      "emma@post.co.uk",  "London",   date(2024, 3, 1)),
    (6, "Farid Aliyev",     None,               None,       date(2024, 3, 10)),
    (7, "Grace Lee",        "grace@web.de",     "Berlin",   date(2024, 3, 15)),
]
products_rows = [
    # product_id, product_name, category, price
    (101, "Laptop",       "Electronics", 1200.0),
    (102, "Headphones",   "Electronics",  150.0),
    (103, "Coffee Maker", "Kitchen",       85.0),
    (104, "Blender",      "Kitchen",       60.0),
    (105, "Novel",        "Books",         15.0),
    (106, "Cookbook",     "Books",         25.0),
    (107, "Desk Lamp",    "Home",          40.0),
]
orders_rows = [
    # order_id, customer_id, product_id, quantity, order_date, status
    (1001, 1,    101, 1, date(2024, 1, 10), "delivered"),
    (1002, 1,    105, 3, date(2024, 1, 10), "delivered"),
    (1003, 2,    102, 2, date(2024, 1, 25), "delivered"),
    (1004, 3,    103, 1, date(2024, 2, 3),  "cancelled"),
    (1005, 2,    106, 1, date(2024, 2, 5),  "delivered"),
    (1006, 4,    104, 2, date(2024, 2, 15), "shipped"),
    (1007, 1,    102, 1, date(2024, 2, 20), "delivered"),
    (1008, 5,    101, 1, date(2024, 3, 2),  "shipped"),
    (1009, 3,    105, 4, date(2024, 3, 5),  "delivered"),
    (1010, 4,    106, 2, date(2024, 3, 8),  "cancelled"),
    (1011, None, 104, 1, date(2024, 3, 9),  "delivered"),
    (1012, 6,    103, 2, date(2024, 3, 12), "shipped"),
    (1013, 2,    101, 1, date(2024, 3, 18), "delivered"),
    (1014, 5,    105, 5, date(2024, 3, 20), "delivered"),
    (1015, 1,    103, 1, date(2024, 3, 25), "shipped"),
    (1016, 3,    108, 1, date(2024, 3, 28), "delivered"),
]

# Write the tables as Parquet files and read them back, the way real data arrives.
# data/shop belongs to this lab: it is rebuilt every time you run this cell.
shutil.rmtree("data/shop", ignore_errors=True)
for table, rows, schema in [
    ("customers", customers_rows, "customer_id INT, name STRING, email STRING, city STRING, signup_date DATE"),
    ("products",  products_rows,  "product_id INT, product_name STRING, category STRING, price DOUBLE"),
    ("orders",    orders_rows,    "order_id INT, customer_id INT, product_id INT, quantity INT, order_date DATE, status STRING"),
]:
    spark.createDataFrame(rows, schema).coalesce(1).write.parquet(f"data/shop/{table}")

customers = spark.read.parquet("data/shop/customers")
products = spark.read.parquet("data/shop/products")
orders = spark.read.parquet("data/shop/orders")

# website events (two rows were loaded twice)
events = spark.createDataFrame([
    # customer_id, event_time, page
    (1, "2024-03-01 09:00:00", "home"),
    (1, "2024-03-01 09:10:00", "product"),
    (1, "2024-03-01 09:10:00", "product"),
    (1, "2024-03-01 11:00:00", "home"),
    (1, "2024-03-02 10:00:00", "cart"),
    (1, "2024-03-03 10:30:00", "home"),
    (1, "2024-03-05 08:00:00", "home"),
    (2, "2024-03-01 20:00:00", "home"),
    (2, "2024-03-01 20:20:00", "product"),
    (2, "2024-03-01 20:45:00", "cart"),
    (2, "2024-03-04 18:00:00", "home"),
    (2, "2024-03-05 18:00:00", "home"),
    (2, "2024-03-05 18:00:00", "home"),
    (3, "2024-03-02 12:00:00", "product"),
    (3, "2024-03-03 12:00:00", "product"),
    (3, "2024-03-04 12:00:00", "cart"),
], "customer_id INT, event_time STRING, page STRING").withColumn("event_time", F.to_timestamp("event_time"))

print("Spark", spark.version, "| Spark UI:", spark.sparkContext.uiWebUrl)

Look at the tables:

In [ ]:
customers.show()
products.show()
orders.show()
events.show()

## Problems

**Q1 · `etl(products)`** Show the most expensive product of each category: category, product_name and price.

Expected output:

```
+-----------+------------+------+
|   category|product_name| price|
+-----------+------------+------+
|      Books|    Cookbook|  25.0|
|Electronics|      Laptop|1200.0|
|       Home|   Desk Lamp|  40.0|
|    Kitchen|Coffee Maker|  85.0|
+-----------+------------+------+
```

<details><summary>Hint</summary>

Number the rows of each category with `row_number`, then keep number 1. This is the classic top-N per group.

</details>

In [ ]:
def etl(products):
    # Write code here: return a DataFrame
    return products

etl(products).show()

**Q2 · `etl(orders)`** For every order of customers 1 and 3, show how many days passed since the same customer's previous order (`days_since_prev`, empty for the first order). Show customer_id, order_id, order_date and days_since_prev.

Expected output:

```
+-----------+--------+----------+---------------+
|customer_id|order_id|order_date|days_since_prev|
+-----------+--------+----------+---------------+
|          1|    1001|2024-01-10|           NULL|
|          1|    1002|2024-01-10|              0|
|          1|    1007|2024-02-20|             41|
|          1|    1015|2024-03-25|             34|
|          3|    1004|2024-02-03|           NULL|
|          3|    1009|2024-03-05|             31|
|          3|    1016|2024-03-28|             23|
+-----------+--------+----------+---------------+
```

<details><summary>Hint</summary>

`F.lag("order_date").over(w)` is the value of the previous row; `F.datediff(end, start)`.

</details>

In [ ]:
def etl(orders):
    # Write code here: return a DataFrame
    return orders

etl(orders).show()

**Q3 · `etl(orders)`** Running total: for customers 1 and 2, show customer_id, order_id, order_date, quantity and `items_so_far` (the items of this order and all earlier orders of the customer).

Expected output:

```
+-----------+--------+----------+--------+------------+
|customer_id|order_id|order_date|quantity|items_so_far|
+-----------+--------+----------+--------+------------+
|          1|    1001|2024-01-10|       1|           1|
|          1|    1002|2024-01-10|       3|           4|
|          1|    1007|2024-02-20|       1|           5|
|          1|    1015|2024-03-25|       1|           6|
|          2|    1003|2024-01-25|       2|           2|
|          2|    1005|2024-02-05|       1|           3|
|          2|    1013|2024-03-18|       1|           4|
+-----------+--------+----------+--------+------------+
```

<details><summary>Hint</summary>

`F.sum("quantity").over(w)` with `rowsBetween(Window.unboundedPreceding, Window.currentRow)`.

</details>

In [ ]:
def etl(orders):
    # Write code here: return a DataFrame
    return orders

etl(orders).show()

**Q4 · `etl(orders, customers)`** Rank the customers by their number of orders. Customers with the same number share a rank, and no rank is skipped after a tie. Show name, orders and rank, sorted by rank, then name.

Expected output:

```
+----------------+------+----+
|            name|orders|rank|
+----------------+------+----+
|    Aziz Karimov|     4|   1|
|   Bella Schmidt|     3|   2|
|     Carlos Ruiz|     3|   2|
|Dilnoza Rahimova|     2|   3|
|     Emma Wilson|     2|   3|
|    Farid Aliyev|     1|   4|
+----------------+------+----+
```

<details><summary>Hint</summary>

`F.dense_rank()` gives 1, 2, 2, 3; `F.rank()` would give 1, 2, 2, 4.

</details>

In [ ]:
def etl(orders, customers):
    # Write code here: return a DataFrame
    return orders

etl(orders, customers).show()

**Q5 · `etl(orders, products)`** Show the top 2 products of each category by items delivered (ties: the smaller product_id first). Columns: category, product_name, items and rank, sorted by category, then rank.

Expected output:

```
+-----------+------------+-----+----+
|   category|product_name|items|rank|
+-----------+------------+-----+----+
|      Books|       Novel|   12|   1|
|      Books|    Cookbook|    1|   2|
|Electronics|  Headphones|    3|   1|
|Electronics|      Laptop|    2|   2|
|    Kitchen|     Blender|    1|   1|
+-----------+------------+-----+----+
```

<details><summary>Hint</summary>

Aggregate per product first, then rank inside each category and keep ranks 1 and 2.

</details>

In [ ]:
def etl(orders, products):
    # Write code here: return a DataFrame
    return orders

etl(orders, products).show()

**Q6 · `etl(orders)`** Show each customer's latest order: customer_id, order_id, order_date and status (on the same day, the larger order_id). Leave out the guest order.

Expected output:

```
+-----------+--------+----------+---------+
|customer_id|order_id|order_date|   status|
+-----------+--------+----------+---------+
|          1|    1015|2024-03-25|  shipped|
|          2|    1013|2024-03-18|delivered|
|          3|    1016|2024-03-28|delivered|
|          4|    1010|2024-03-08|cancelled|
|          5|    1014|2024-03-20|delivered|
|          6|    1012|2024-03-12|  shipped|
+-----------+--------+----------+---------+
```

<details><summary>Hint</summary>

`row_number` over a window ordered by date descending; keep number 1.

</details>

In [ ]:
def etl(orders):
    # Write code here: return a DataFrame
    return orders

etl(orders).show()

**Q7 · `etl(events)`** Split the events (duplicates removed) of customers 1 and 2 into sessions: a new session starts when more than 30 minutes passed since the customer's previous event. Show customer_id, event_time, page and session (1, 2, ... per customer).

Expected output:

```
+-----------+-------------------+-------+-------+
|customer_id|         event_time|   page|session|
+-----------+-------------------+-------+-------+
|          1|2024-03-01 09:00:00|   home|      1|
|          1|2024-03-01 09:10:00|product|      1|
|          1|2024-03-01 11:00:00|   home|      2|
|          1|2024-03-02 10:00:00|   cart|      3|
|          1|2024-03-03 10:30:00|   home|      4|
|          1|2024-03-05 08:00:00|   home|      5|
|          2|2024-03-01 20:00:00|   home|      1|
|          2|2024-03-01 20:20:00|product|      1|
|          2|2024-03-01 20:45:00|   cart|      1|
|          2|2024-03-04 18:00:00|   home|      2|
|          2|2024-03-05 18:00:00|   home|      3|
+-----------+-------------------+-------+-------+
```

<details><summary>Hint</summary>

Mark the first event of each session with 1 (others 0), then a running sum of the marks numbers the sessions.

</details>

In [ ]:
def etl(events):
    # Write code here: return a DataFrame
    return events

etl(events).show()

**Q8 · `etl(events)`** What is each customer's longest streak of consecutive days with at least one event? Show customer_id and longest_streak.

Expected output:

```
+-----------+--------------+
|customer_id|longest_streak|
+-----------+--------------+
|          1|             3|
|          2|             2|
|          3|             3|
+-----------+--------------+
```

<details><summary>Hint</summary>

Gaps and islands: number each customer's distinct days with `row_number` and subtract that number from the date. Days of one streak get the same result, so group by it.

</details>

In [ ]:
def etl(events):
    # Write code here: return a DataFrame
    return events

etl(events).show()

## Timed set: 6 problems in 65 minutes
Set a timer. No hints: write each `etl` from scratch, as in an interview. Easy 5 minutes, medium 10, hard 15.

**Q9 · `etl(orders, products)`** (easy, 5 min) the products that were ordered but never delivered. Columns: product_id, product_name.

Expected output:

```
+----------+------------+
|product_id|product_name|
+----------+------------+
|       103|Coffee Maker|
+----------+------------+
```

In [ ]:
def etl(orders, products):
    # Write code here: return a DataFrame
    return orders

etl(orders, products).show()

**Q10 · `etl(orders, customers)`** (medium, 10 min) in each city, the customer with the most orders (ties: the smaller customer_id). Customers without a city are left out. Columns: city, name, orders.

Expected output:

```
+--------+-------------+------+
|    city|         name|orders|
+--------+-------------+------+
|  Berlin|Bella Schmidt|     3|
|  London|  Emma Wilson|     2|
|  Madrid|  Carlos Ruiz|     3|
|Tashkent| Aziz Karimov|     4|
+--------+-------------+------+
```

In [ ]:
def etl(orders, customers):
    # Write code here: return a DataFrame
    return orders

etl(orders, customers).show()

**Q11 · `etl(orders, products)`** (medium, 10 min) the delivered revenue (quantity × price) per month and its growth against the previous month in percent (1 decimal, empty for the first month). Columns: month, revenue, growth_pct.

Expected output:

```
+-----+-------+----------+
|month|revenue|growth_pct|
+-----+-------+----------+
|    1| 1545.0|      NULL|
|    2|  175.0|     -88.7|
|    3| 1395.0|     697.1|
+-----+-------+----------+
```

In [ ]:
def etl(orders, products):
    # Write code here: return a DataFrame
    return orders

etl(orders, products).show()

**Q12 · `etl(orders)`** (medium, 10 min) the customers whose first order (by date, then order_id) was delivered but whose latest order was not. Columns: customer_id, first_status, last_status.

Expected output:

```
+-----------+------------+-----------+
|customer_id|first_status|last_status|
+-----------+------------+-----------+
|          1|   delivered|    shipped|
+-----------+------------+-----------+
```

In [ ]:
def etl(orders):
    # Write code here: return a DataFrame
    return orders

etl(orders).show()

**Q13 · `etl(orders)`** (hard, 15 min) for each customer with at least 2 orders, the longest gap in days between two consecutive orders. Columns: customer_id, longest_gap_days.

Expected output:

```
+-----------+----------------+
|customer_id|longest_gap_days|
+-----------+----------------+
|          1|              41|
|          2|              42|
|          3|              31|
|          4|              22|
|          5|              18|
+-----------+----------------+
```

In [ ]:
def etl(orders):
    # Write code here: return a DataFrame
    return orders

etl(orders).show()

**Q14 · `etl(orders, products)`** (hard, 15 min) rank the customers by delivered revenue (1 = highest; guests don't count) and show how much less each one spent than the customer ranked just above. Columns: customer_id, revenue, rank, gap_to_next_better.

Expected output:

```
+-----------+-------+----+------------------+
|customer_id|revenue|rank|gap_to_next_better|
+-----------+-------+----+------------------+
|          2| 1525.0|   1|              NULL|
|          1| 1395.0|   2|             130.0|
|          5|   75.0|   3|            1320.0|
|          3|   60.0|   4|              15.0|
+-----------+-------+----+------------------+
```

In [ ]:
def etl(orders, products):
    # Write code here: return a DataFrame
    return orders

etl(orders, products).show()

## Answers
Try each problem yourself first. Click a problem to open its answer.

<details>
<summary><b>Q1</b> &nbsp;Show the most expensive product of each category: category, product_name and price.</summary>

```python
def etl(products):
    w = Window.partitionBy("category").orderBy(F.col("price").desc())
    return (products.withColumn("rn", F.row_number().over(w))
                    .filter(F.col("rn") == 1)
                    .select("category", "product_name", "price")
                    .orderBy("category"))

etl(products).show()
```

</details>

<details>
<summary><b>Q2</b> &nbsp;For every order of customers 1 and 3, show how many days passed since the same customer's previous order ...</summary>

```python
def etl(orders):
    w = Window.partitionBy("customer_id").orderBy("order_date", "order_id")
    return (orders.filter(F.col("customer_id").isin(1, 3))
                  .withColumn("days_since_prev", F.datediff("order_date", F.lag("order_date").over(w)))
                  .select("customer_id", "order_id", "order_date", "days_since_prev")
                  .orderBy("customer_id", "order_date", "order_id"))

etl(orders).show()
```

</details>

<details>
<summary><b>Q3</b> &nbsp;Running total: for customers 1 and 2, show customer_id, order_id, order_date, quantity and items_so_far (the ...</summary>

```python
def etl(orders):
    w = (Window.partitionBy("customer_id").orderBy("order_date", "order_id")
               .rowsBetween(Window.unboundedPreceding, Window.currentRow))
    return (orders.filter(F.col("customer_id").isin(1, 2))
                  .withColumn("items_so_far", F.sum("quantity").over(w))
                  .select("customer_id", "order_id", "order_date", "quantity", "items_so_far")
                  .orderBy("customer_id", "order_date", "order_id"))

etl(orders).show()
```

</details>

<details>
<summary><b>Q4</b> &nbsp;Rank the customers by their number of orders. Customers with the same number share a rank, and no rank is ...</summary>

```python
def etl(orders, customers):
    counts = orders.groupBy("customer_id").agg(F.count("*").alias("orders"))
    w = Window.orderBy(F.col("orders").desc())
    return (counts.join(customers, "customer_id")
                  .withColumn("rank", F.dense_rank().over(w))
                  .select("name", "orders", "rank")
                  .orderBy("rank", "name"))

etl(orders, customers).show()
```

</details>

<details>
<summary><b>Q5</b> &nbsp;Show the top 2 products of each category by items delivered (ties: the smaller product_id first). Columns: ...</summary>

```python
def etl(orders, products):
    items = (orders.filter(F.col("status") == "delivered")
                   .join(products, "product_id")
                   .groupBy("category", "product_id", "product_name")
                   .agg(F.sum("quantity").alias("items")))
    w = Window.partitionBy("category").orderBy(F.col("items").desc(), "product_id")
    return (items.withColumn("rank", F.row_number().over(w))
                 .filter(F.col("rank") <= 2)
                 .select("category", "product_name", "items", "rank")
                 .orderBy("category", "rank"))

etl(orders, products).show()
```

</details>

<details>
<summary><b>Q6</b> &nbsp;Show each customer's latest order: customer_id, order_id, order_date and status (on the same day, the larger ...</summary>

```python
def etl(orders):
    w = Window.partitionBy("customer_id").orderBy(F.col("order_date").desc(), F.col("order_id").desc())
    return (orders.filter(F.col("customer_id").isNotNull())
                  .withColumn("rn", F.row_number().over(w))
                  .filter(F.col("rn") == 1)
                  .select("customer_id", "order_id", "order_date", "status")
                  .orderBy("customer_id"))

etl(orders).show()
```

</details>

<details>
<summary><b>Q7</b> &nbsp;Split the events (duplicates removed) of customers 1 and 2 into sessions: a new session starts when more than ...</summary>

```python
def etl(events):
    w = Window.partitionBy("customer_id").orderBy("event_time")
    gap_minutes = (F.unix_timestamp("event_time") - F.unix_timestamp(F.lag("event_time").over(w))) / 60
    return (events.dropDuplicates()
                  .filter(F.col("customer_id").isin(1, 2))
                  .withColumn("new_session", F.when(gap_minutes.isNull() | (gap_minutes > 30), 1).otherwise(0))
                  .withColumn("session", F.sum("new_session").over(w))
                  .select("customer_id", "event_time", "page", "session")
                  .orderBy("customer_id", "event_time"))

etl(events).show()
```

</details>

<details>
<summary><b>Q8</b> &nbsp;What is each customer's longest streak of consecutive days with at least one event? Show customer_id and ...</summary>

```python
def etl(events):
    days = events.select("customer_id", F.to_date("event_time").alias("day")).distinct()
    w = Window.partitionBy("customer_id").orderBy("day")
    streaks = (days.withColumn("group", F.date_sub("day", F.row_number().over(w)))
                   .groupBy("customer_id", "group").count())
    return (streaks.groupBy("customer_id")
                   .agg(F.max("count").alias("longest_streak"))
                   .orderBy("customer_id"))

etl(events).show()
```

</details>

<details>
<summary><b>Q9</b> &nbsp;(easy, 5 min) the products that were ordered but never delivered. Columns: product_id, product_name.</summary>

```python
def etl(orders, products):
    ordered = orders.select("product_id").distinct()
    delivered = orders.filter(F.col("status") == "delivered").select("product_id")
    return (ordered.join(delivered, "product_id", "left_anti")
                   .join(products, "product_id")
                   .select("product_id", "product_name")
                   .orderBy("product_id"))

etl(orders, products).show()
```

</details>

<details>
<summary><b>Q10</b> &nbsp;(medium, 10 min) in each city, the customer with the most orders (ties: the smaller customer_id). Customers ...</summary>

```python
def etl(orders, customers):
    counts = orders.groupBy("customer_id").agg(F.count("*").alias("orders"))
    w = Window.partitionBy("city").orderBy(F.desc("orders"), "customer_id")
    return (customers.filter(F.col("city").isNotNull())
                     .join(counts, "customer_id")
                     .withColumn("rn", F.row_number().over(w))
                     .filter(F.col("rn") == 1)
                     .select("city", "name", "orders")
                     .orderBy("city"))

etl(orders, customers).show()
```

</details>

<details>
<summary><b>Q11</b> &nbsp;(medium, 10 min) the delivered revenue (quantity × price) per month and its growth against the previous month ...</summary>

```python
def etl(orders, products):
    monthly = (orders.filter(F.col("status") == "delivered")
                     .join(products, "product_id")
                     .groupBy(F.month("order_date").alias("month"))
                     .agg(F.sum(F.col("quantity") * F.col("price")).alias("revenue")))
    previous = F.lag("revenue").over(Window.orderBy("month"))
    return (monthly.withColumn("growth_pct", F.round((F.col("revenue") - previous) / previous * 100, 1))
                   .orderBy("month"))

etl(orders, products).show()
```

</details>

<details>
<summary><b>Q12</b> &nbsp;(medium, 10 min) the customers whose first order (by date, then order_id) was delivered but whose latest ...</summary>

```python
def etl(orders):
    by_time = F.struct("order_date", "order_id")
    return (orders.filter(F.col("customer_id").isNotNull())
                  .groupBy("customer_id")
                  .agg(F.min_by("status", by_time).alias("first_status"), F.max_by("status", by_time).alias("last_status"))
                  .filter((F.col("first_status") == "delivered") & (F.col("last_status") != "delivered"))
                  .orderBy("customer_id"))

etl(orders).show()
```

</details>

<details>
<summary><b>Q13</b> &nbsp;(hard, 15 min) for each customer with at least 2 orders, the longest gap in days between two consecutive ...</summary>

```python
def etl(orders):
    w = Window.partitionBy("customer_id").orderBy("order_date", "order_id")
    return (orders.filter(F.col("customer_id").isNotNull())
                  .withColumn("gap_days", F.datediff("order_date", F.lag("order_date").over(w)))
                  .groupBy("customer_id")
                  .agg(F.max("gap_days").alias("longest_gap_days"))
                  .filter(F.col("longest_gap_days").isNotNull())
                  .orderBy("customer_id"))

etl(orders).show()
```

</details>

<details>
<summary><b>Q14</b> &nbsp;(hard, 15 min) rank the customers by delivered revenue (1 = highest; guests don't count) and show how much ...</summary>

```python
def etl(orders, products):
    revenue = (orders.filter((F.col("status") == "delivered") & F.col("customer_id").isNotNull())
                     .join(products, "product_id")
                     .groupBy("customer_id")
                     .agg(F.sum(F.col("quantity") * F.col("price")).alias("revenue")))
    w = Window.orderBy(F.desc("revenue"))
    return (revenue.withColumn("rank", F.rank().over(w))
                   .withColumn("gap_to_next_better", F.lag("revenue").over(w) - F.col("revenue"))
                   .orderBy("rank"))

etl(orders, products).show()
```

</details>